# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

**Solution**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils import resample

**1.** Distribution of the target variable

In [ ]:
print(fraud["fraud"].value_counts())
print()
print(fraud["fraud"].value_counts(normalize=True))

Only about **8.7%** of transactions are fraud, so **yes — this is a strongly imbalanced dataset.**

**2.** Train a Logistic Regression

In [ ]:
X = fraud.drop(columns=["fraud"])
y = fraud["fraud"].astype(int)

# split first (stratify keeps the same fraud ratio in train and test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

**3.** Evaluate — on imbalanced data **accuracy is misleading**, so the key metric is **recall / precision on the fraud class (1)** plus the confusion matrix.

In [ ]:
pred = logreg.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

**4.** Oversampling — duplicate minority (fraud) rows on the **training set only**, then re-train and evaluate on the original test set.

In [ ]:
train = X_train.copy()
train["fraud"] = y_train.values
fraud_rows = train[train["fraud"] == 1]
legit_rows = train[train["fraud"] == 0]

fraud_oversampled = resample(fraud_rows, replace=True,
                             n_samples=len(legit_rows), random_state=42)
train_over = pd.concat([legit_rows, fraud_oversampled])
print(train_over["fraud"].value_counts())

X_train_over = train_over.drop(columns=["fraud"])
y_train_over = train_over["fraud"]

logreg_over = LogisticRegression(max_iter=1000)
logreg_over.fit(X_train_over, y_train_over)
pred_over = logreg_over.predict(X_test)          # evaluate on ORIGINAL test set
print(classification_report(y_test, pred_over))

**5.** Undersampling — drop majority (legit) rows down to the minority size.

In [ ]:
legit_undersampled = resample(legit_rows, replace=False,
                              n_samples=len(fraud_rows), random_state=42)
train_under = pd.concat([legit_undersampled, fraud_rows])
print(train_under["fraud"].value_counts())

X_train_under = train_under.drop(columns=["fraud"])
y_train_under = train_under["fraud"]

logreg_under = LogisticRegression(max_iter=1000)
logreg_under.fit(X_train_under, y_train_under)
pred_under = logreg_under.predict(X_test)
print(classification_report(y_test, pred_under))

**6.** SMOTE — create *synthetic* minority rows (needs `imbalanced-learn`).

In [ ]:
# !pip install imbalanced-learn   # run once if not installed
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print(pd.Series(y_train_sm).value_counts())

logreg_smote = LogisticRegression(max_iter=1000)
logreg_smote.fit(X_train_sm, y_train_sm)
pred_smote = logreg_smote.predict(X_test)
print(classification_report(y_test, pred_smote))

**Conclusion**\n\nCompare the **recall/precision for class 1 (fraud)** across baseline / oversample / undersample / SMOTE. All three balancing methods **raise fraud recall** (we catch more fraud) but usually **lower precision** (more false alarms). **SMOTE** normally gives the best recall/precision balance, because it adds synthetic variety instead of exact copies. Golden rule: always resample the **training set only** and evaluate on the **untouched test set**.